In [11]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import geopandas as gpd
import rioxarray as rioxr
from shapely import box

import pystac_client
import planetary_computer

from IPython.display import Image


sys.path.append("../utils")

pd.set_option("display.max_columns", None)


In [12]:
in19 = gpd.read_file(
    "/capstone/wildfire_prep/data/inspections_data/cleaned_status/inspections_2019.geojson"
)
in20 = gpd.read_file(
    "/capstone/wildfire_prep/data/inspections_data/cleaned_status/inspections_2020.geojson"
)
in21 = gpd.read_file(
    "/capstone/wildfire_prep/data/inspections_data/cleaned_status/inspections_2021.geojson"
)
in22 = gpd.read_file(
    "/capstone/wildfire_prep/data/inspections_data/cleaned_status/inspections_2022.geojson"
)
in23 = gpd.read_file(
    "/capstone/wildfire_prep/data/inspections_data/cleaned_status/inspections_2023.geojson"
)


In [13]:
# structuret

colnames_23 = {
    "system_created_at": "system_cre",
    "address_suite": "address_su",
    "address_thoroughfare": "address_th",
    "address_locality": "address_lo",
    "address_sub_admin_area": "address__2",
    "address_postal_code": "address_po",
    "address_full": "address_fu",
    "inspectiondate_calculate": "inspection",
    "structuretype": "structuret",
    "address_admin_area": "address_ad",
    "address_country": "address_co",
}

in23 = in23.rename(columns=colnames_23)


In [14]:
inspections = pd.concat([in19, in20, in21, in22, in23], axis=0, ignore_index=True)


/tmp/ipykernel_572262/1430165923.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  inspections = pd.concat([in19, in20, in21, in22, in23], axis=0, ignore_index=True)


In [15]:
in23.head(20)

,fulcrum_id,created_at,updated_at,created_by,updated_by,system_cre,system_updated_at,version,status,project,assigned_to,latitude,longitude,report_title,globalid,keyid,propertystatus,inspectionstatus,inspectorfirstname,inspectorlastname,prevention_inspectorfirstname,prevention_inspectorfirstname_other,prevention_inspectorlastname,prevention_inspectorlastname_other,inspectorposition,inspection,addressvisible,address_sub_thoroughfare,address_th,address_su,address_lo,address__2,address_ad,address_po,address_co,address_fu,calfireunit,county,community,community_other,battalion,enginenumber,stationname,shift,shift_julian,accessegress,occupanthome,deliverynotificationmethod,inspectionhours,a_removebranchesfromstovepipe,b_removeleavesneedlesveg,c_removedeaddyingtrees,d_removedeaddyinggrassplants,e_removeflammablegroundcover,f_removeflammablevegetation,g_relocateexposedwoodpiles,h_cutannualgrassesforbs,i_removefuelsusingctguidelines,j_exposedwoodpiles,k_removedeaddyingwoodyfuels,l_removelogstumpsembeddedsoil,m_outbuildingsliquidpropanegas,n_displayaddresscontrasting,o_stovepipemetalscreenopenings,recommendclearvegetation,waterstora,water_source,can_engine_access_water_source,water_storage_size_stored_water_on_individual_parcels_only,water_comments,structuret,structurehabitable,roofconstruction,eaves,ventscreen,exteriorsiding,windowpane,deckporchgrade,deckporchelevated,patiocovercarport,fenceattachedtostructure,propanetankdistance,utilitymiscstructuredistance,utilitymiscstucturecount,nonhabitableoutbuildings,escalatetocoordinator,reinspectiondate,citationnumber,number_of_dead_trees_within_300_ft_of_residence,gatecode,comments,photoid,photoid_caption,photoid_url,le100number,inspectioncount,coredata,firescopeid,apn,yearbuilt,siteaddress,numberofstructures,appenddate,editstatus,calculatedglobalid,calculateddate,calculateduid,textfield1,textfield2,numberfield1,numberfield2,previousyrinspecteddata,previousyrinspectionstatus,calculatededitor,calculatededitdate,creationdate,editdate,creator,editor,Date,geometry
0,48c58877-f0ce-438d-ac87-6434a2c15fe8,2022-02-13 16:00:00,2023-06-05 17:00:00,gis@sbcfire.com,gis@sbcfire.com,2022-02-14 10:44:17,2023-12-06 08:54:05,13,Compliant,None,None,34.461625,-119.772555,Defensible Space Inspection Report,None,0.0,None,None,None,None,Charlotte,None,Endicott,None,DSI,2023-06-05,Yes - Without Reflective,4440,Shadow Hills Cir,None,Santa Barbara,Santa Barbara,CA,93105,US,4440 Shadow Hills Cir Santa Barbara Santa Barb...,SBC,SBA,San Marcos Foothills,None,1,NaN,13,A,157,Yes,No,Mailed,15,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,yes,Hydrant- Municipal/Public Water Supply,yes,None,None,Multi Family Residence Multi Story,Habitable,Tile,Enclosed,"Mesh Screen <= 1/8""",Other,Single Pane,Masonry/Concrete,Masonry/Concrete,Non Combustible,No Fence,Not Applicable,None,0,0,None,NaT,None,0,None,None,"eb994fe9-6273-444c-95b7-c3633b0876f5,de1dc62a-...",",,",https://web.fulcrumapp.com/photos/view?photos=...,None,None,None,None,None,NaN,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,2022-02-14,2022-02-14,None,None,2023-06-05,POINT (-119.77256 34.46163)
1,f351800c-fc5c-4b79-bf24-a41775b74d7c,2022-04-04 17:00:00,2023-10-06 09:47:59,gis@sbcfire.com,sbc.dsp.inspector@sbcfire.com,2022-04-05 13:47:06,2023-10-06 09:55:52,8,Compliant,None,None,34.672591,-120.108711,Defensible Space Inspection Report,None,0.0,None,None,None,None,None,None,Gailey,None,Prevention,2023-10-06,Yes - Without Reflective,3110 A,Acampo Rd,None,Santa Ynez,Santa Barbara,CA,93441,US,3110 A Acampo Rd Santa Ynez Santa Barbara CA 9...,SBC,SBA,Los Olivos,None,3,32.0,Santa Ynez Valley (Outside Engine Co. Area),B,279,Yes,No,Hardcopy,15,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,yes,Hydrant- Municipal/Public Water Supply,yes,None,None,Single Family Residence Single Story,Habitable,Tile,Unenclosed,No Vents,Stucco/Brick/Cement,Single Pane,Masonry/Concrete,No Deck/Porch,Combusti

In [16]:
inspections["inspection_id"] = np.arange(1, len(inspections) + 1)

/Users/ryangreen/.conda/envs/prg/lib/python3.12/site-packages/geopandas/geodataframe.py:1819: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)


In [17]:
inspections = inspections[["inspection_id", "structuret"]]


In [18]:
structuretype_codes = (
    inspections[["structuret"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

structuretype_codes["structure_code"] = range(100, 100 + len(structuretype_codes))

structuretype_codes


,structuret,structure_code
0,Utility or Miscellaneous Structure > 120 sqft,100
1,Single Family Residence Multi Story,101
2,Single Family Residence Single Story,102
3,Unable to Assess,103
4,Commercial Building Single Story,104
5,Infrastructure (Essential Services),105
6,Mobile Home Double Wide,106
7,Multi Family Residence Single Story,107
8,Mobile Home Single Wide,108
9,Church,109


In [19]:
structuretype_codes.to_csv(
    "/capstone/wildfire_prep/data/metadata/structuretype_codes.csv",
    index=False,
)


In [27]:
inspections

,inspection_id,structuret
0,1,Utility or Miscellaneous Structure > 120 sqft
1,2,Single Family Residence Multi Story
2,3,Single Family Residence Multi Story
3,4,Utility or Miscellaneous Structure > 120 sqft
4,5,Single Family Residence Multi Story
...,...,...
67575,67576,Single Family Residence Multi Story
67576,67577,None
67577,67578,Mobile Home Double Wide
67578,67579,Mobile Home Double Wide


In [37]:
inspections = inspections.merge(
    structuretype_codes[["structuret", "structure_code"]],
    on="structuret",
    how="left",
).drop(columns=["structuret"])
inspections

,inspection_id,structure_code
0,1,100
1,2,101
2,3,101
3,4,100
4,5,101
...,...,...
67575,67576,101
67576,67577,116
67577,67578,106
67578,67579,106


In [38]:
inspections.to_csv(
    "/capstone/wildfire_prep/data/PUZZLE_PIECES/inspections_id_structure_type.csv",
    index=False,
)
